In [ ]:
import numpy as np
import cellrank as cr
import scanpy as sc
import scvelo as scv
import matplotlib.pyplot as plt
import matplotlib
scv.settings.verbosity = 3
scv.settings.set_figure_params("scvelo")
sc.settings.set_figure_params(frameon=False, dpi=100)
cr.settings.verbosity = 2
import warnings
warnings.simplefilter("ignore", category=UserWarning)

In [ ]:
adata_CD4T = sc.read("../data/adata_CD4T_adata.h5ad")
adata_CD4T = adata_CD4T.raw.to_adata()
adata_CD4T

In [3]:
import numpy as np
adata_CD4T.uns["CD4T_celltype_colors"] = np.array(['#941456', '#e069a6', '#a56ba7', '#e0a7c8', '#1f577b'], dtype=object)

In [ ]:
adata_CD4T_magic = adata_CD4T.copy()
adata_CD4T_magic.X = adata_CD4T.uns['magic_imputed_data_CD4T']
adata_CD4T_magic

**ALL**

In [ ]:
pk_CD4T_all = cr.kernels.PseudotimeKernel(adata_CD4T, time_key="palantir_pseudotime")
pk_CD4T_all.compute_transition_matrix()
print(pk_CD4T_all)

**HC_pk**

In [6]:
import numpy as np
np.random.seed(0)

In [7]:
adata_CD4T_HC = adata_CD4T[adata_CD4T.obs['group']=='HC'].copy()
adata_CD4T_HC = adata_CD4T_HC[adata_CD4T_HC.obs_names[np.random.permutation(1500)]]

In [ ]:
adata_CD4T_HC

In [ ]:
sc.pp.neighbors(adata_CD4T_HC, 
                n_neighbors=30,
                n_pcs=15,
                use_rep='X_scVI')

In [ ]:
pk_CD4T_HC = cr.kernels.PseudotimeKernel(adata_CD4T_HC, time_key="palantir_pseudotime")
pk_CD4T_HC.compute_transition_matrix()
print(pk_CD4T_HC)

**TED_typeI_pk**

In [ ]:
adata_CD4T_TED_typeI = adata_CD4T[adata_CD4T.obs['group']=='TED_typeI'].copy()
adata_CD4T_TED_typeI = adata_CD4T_TED_typeI[adata_CD4T_TED_typeI.obs_names[np.random.permutation(1500)]]

In [ ]:
sc.pp.neighbors(adata_CD4T_TED_typeI, 
                n_neighbors=30,
                n_pcs=15,
                use_rep='X_scVI')

In [ ]:
pk_CD4T_TED_typeI = cr.kernels.PseudotimeKernel(adata_CD4T_TED_typeI, time_key="palantir_pseudotime")
pk_CD4T_TED_typeI.compute_transition_matrix()
print(pk_CD4T_TED_typeI)

**TED_typeII_pk**

In [ ]:
adata_CD4T_TED_typeII = adata_CD4T[adata_CD4T.obs['group']=='TED_typeII'].copy()
adata_CD4T_TED_typeII = adata_CD4T_TED_typeII[adata_CD4T_TED_typeII.obs_names[np.random.permutation(1500)]]

In [ ]:
sc.pp.neighbors(adata_CD4T_TED_typeII, 
                n_neighbors=30,
                n_pcs=15,
                use_rep='X_scVI')

In [ ]:
pk_CD4T_TED_typeII = cr.kernels.PseudotimeKernel(adata_CD4T_TED_typeII, time_key="palantir_pseudotime")
pk_CD4T_TED_typeII.compute_transition_matrix()
print(pk_CD4T_TED_typeII)

**projection**

In [ ]:
matplotlib.use('Agg')
pk_CD4T_all.plot_projection(basis="X_diffmap", components='0,1',
                            palette=['#941456', '#e069a6', '#a56ba7', '#e0a7c8', '#1f577b'],
                            legend_loc='right margin',recompute=False)
plt.savefig('./output/CD4T/1.pk_CD4T_TED_all.svg', dpi=600)
plt.show()

In [ ]:
matplotlib.use('Agg')
pk_CD4T_HC.plot_projection(basis="X_diffmap", components='0,1',
                           palette=['#941456', '#e069a6', '#a56ba7', '#e0a7c8', '#1f577b'],
                           legend_loc='right margin', recompute=False)
plt.savefig('./output/CD4T/1.pk_CD4T_TED_HC.svg', dpi=600)
plt.show()

In [ ]:
matplotlib.use('Agg')
pk_CD4T_TED_typeI.plot_projection(basis="X_diffmap",  components='0,1',
                                  palette=['#941456', '#e069a6', '#a56ba7', '#e0a7c8', '#1f577b'],
                                  legend_loc='right margin',recompute=False)
plt.savefig('./output/CD4T/1.pk_CD4T_TED_typeI.svg', dpi=600)
plt.show()

In [ ]:
matplotlib.use('Agg')
pk_CD4T_TED_typeII.plot_projection(basis="X_diffmap", components='0,1',
                                   palette=['#941456', '#e069a6', '#a56ba7', '#e0a7c8', '#1f577b'],
                                   legend_loc='right margin',recompute=False)
plt.savefig('./output/CD4T/1.pk_CD4T_TED_typeII.svg', dpi=600)
plt.show()

**All**

In [ ]:
g_CD4T_all = cr.estimators.GPCCA(pk_CD4T_all) 
print(g_CD4T_all)

In [ ]:
%matplotlib inline
g_CD4T_all.compute_schur()
g_CD4T_all.plot_spectrum(real_only=True, s=300,figsize = [8,4])

In [ ]:
g_CD4T_all.compute_macrostates(n_states=4, cluster_key="CD4T_celltype")
g_CD4T_all.plot_macrostates(basis="X_diffmap",which="all", components='0,1',legend_loc="right", s=100,save='CD4T_all_macrostates.pdf')
g_CD4T_all.plot_macrostate_composition(key="CD4T_celltype", figsize=(7, 4),save='CD4T_all_macrostate_composition.pdf')
g_CD4T_all.plot_coarse_T(annotate=True,save='CD4T_all_plot_coarse_T.pdf')

In [ ]:
g_CD4T_all.set_initial_states(states=["CD4T_c02-LEF1"])
g_CD4T_all.set_terminal_states(states=['CD4T_c04-GNLY', 'CD4T_c05-FOXP3','CD4T_c01-GATA3'])

In [ ]:
g_CD4T_all.plot_macrostates(basis="X_diffmap", which="all", discrete=True, components='0,1',
                            legend_loc="right", s=100,save='CD4T_all_macrostates.svg')
g_CD4T_all.plot_macrostates(basis="X_diffmap", which="initial", discrete=True, components='0,1',
                            legend_loc="right", s=100,save='CD4T_all_initial_macrostates.svg')
g_CD4T_all.plot_macrostates(basis="X_diffmap", which="terminal", discrete=True, components='0,1',
                            legend_loc="right", s=100,save='CD4T_all_terminal_macrostates.svg')

In [ ]:
%matplotlib inline
g_CD4T_all.compute_fate_probabilities()
cr.pl.circular_projection(adata_CD4T, keys=["CD4T_celltype"], figsize = [30,30], s=50,alpha=0.7,
                          legend_loc="right",save='CD4T_all_circular_projection.pdf')

**HC**

In [ ]:
g_CD4T_HC = cr.estimators.GPCCA(pk_CD4T_HC)
print(g_CD4T_HC)

In [ ]:
g_CD4T_HC.compute_macrostates(n_states=5, cluster_key="CD4T_celltype")
g_CD4T_HC.plot_macrostates(basis="X_diffmap",which="all", legend_loc="right", components='0,1',s=100, save='CD4T_HC_macrostates_manul.pdf')

In [ ]:
g_CD4T_HC.set_initial_states(states=["CD4T_c02-LEF1"])
g_CD4T_HC.set_terminal_states(states=['CD4T_c04-GNLY', 'CD4T_c05-FOXP3','CD4T_c01-GATA3_2'])

In [ ]:
%matplotlib inline
g_CD4T_HC.compute_fate_probabilities()
cr.pl.circular_projection(adata_CD4T_HC, keys=["CD4T_celltype"], figsize = [30,30], s=80,alpha=0.7,legend_loc="right", save='CD4T_HC_circular_projection_manul.pdf')

**TED_typeI**

In [ ]:
g_CD4T_TED_typeI = cr.estimators.GPCCA(pk_CD4T_TED_typeI)
print(g_CD4T_TED_typeI)

In [ ]:
g_CD4T_TED_typeI.compute_macrostates(n_states=5, cluster_key="CD4T_celltype")
g_CD4T_TED_typeI.plot_macrostates(basis="X_diffmap",which="all", legend_loc="right", s=100, components='0,1',save='CD4T_TED_typeI_macrostates_manul.pdf')

In [ ]:
g_CD4T_TED_typeI.set_initial_states(states=["CD4T_c02-LEF1"])
g_CD4T_TED_typeI.set_terminal_states(states=['CD4T_c04-GNLY', 'CD4T_c05-FOXP3','CD4T_c01-GATA3_2'])

In [ ]:
%matplotlib inline
g_CD4T_TED_typeI.compute_fate_probabilities()
cr.pl.circular_projection(adata_CD4T_TED_typeI, keys=["CD4T_celltype"], figsize = [30,30], s=80,alpha=0.7,legend_loc="right", save='CD4T_TED_typeI_circular_projection_manul.pdf')

**TED_typeII**

In [ ]:
g_CD4T_TED_typeII = cr.estimators.GPCCA(pk_CD4T_TED_typeII)
print(g_CD4T_TED_typeII)

In [ ]:
g_CD4T_TED_typeII.compute_macrostates(n_states=5, cluster_key="CD4T_celltype")
g_CD4T_TED_typeII.plot_macrostates(basis="X_diffmap",which="all", legend_loc="right", s=100, components='0,1',save='CD4T_TED_typeII_macrostates_manul.pdf')

In [ ]:
g_CD4T_TED_typeII.set_initial_states(states=["CD4T_c02-LEF1"])
g_CD4T_TED_typeII.set_terminal_states(states=['CD4T_c04-GNLY', 'CD4T_c05-FOXP3','CD4T_c01-GATA3_1'])

In [ ]:
%matplotlib inline
g_CD4T_TED_typeII.compute_fate_probabilities()
cr.pl.circular_projection(adata_CD4T_TED_typeII, keys=["CD4T_celltype"], figsize = [30,30], s=80,alpha=0.7,legend_loc="right", save='CD4T_TED_typeII_circular_projection_manul.pdf')

In [ ]:
adata_CD4T.layers["magic_imputed_data"] = adata_CD4T_magic.X.copy()

In [ ]:
model = cr.models.GAM(adata_CD4T, n_knots=6)

In [ ]:
cr.pl.gene_trends(
    adata_CD4T,
    model=model,
    data_key="magic_imputed_data",
    genes=["TBX21", "GATA3","ARID5B","LEF1","FOXP3"],
    same_plot=True,
    ncols=2,
    time_key="palantir_pseudotime",
    hide_cells=True,
    weight_threshold=(1e-3, 1e-3), save='CD4T_gene_trends_palantir_pseudotime.pdf'
)